# Joint-use demo — fetal mortality rate, two stratifications

Demonstrates the U.S. Harmonized Vital Statistics resource's joint-use design by
computing fetal mortality rates that need both the fetal-death numerator and the
natality denominator, stratified two ways:

- **Section A** — 2022 by maternal age band, validated byte-exact against
  *NVSR 73-09* Table 4 (8 cells, all PASS).
- **Section B** — 2017 by maternal race (last year `maternal_race_bridged` is
  available in both products; NCHS dropped MBRACE from 2018+ public-use files).
  Machinery demonstration; *NVSR* cell-level validation is deferred to the paper
  companion notebook.

**Canonical analytic filters** (applied identically in numerator and denominator):

| Product | Filter |
|---|---|
| Natality | `restatus != 4` (int) — U.S. residents |
| Linked birth–infant death | `restatus != 4` (int) — U.S. residents |
| Fetal death | `tabulation_flag == '2' AND residence_status != '4'` (string, see dtype note below) — NVSR-comparable >=20wk resident |

**Dtype note.** The fetal-death v2.0.0 parquet stores `tabulation_flag`,
`residence_status`, `maternal_age`, `maternal_race_bridged`, and `hispanic_origin`
as `object` (string), whereas `fetal_death/harmonized_schema.csv` documents them
as `int`. This notebook uses string literals on the fetal-death side and integer
literals on the natality side. The schema-vs-data drift is logged in FIX_LOG.md
and a future task will reconcile the schema docs (a schema-version bump per the
operating protocol's §9 anti-pattern 6).

In [1]:
import pandas as pd
import sys
from pathlib import Path

# Add repo root to sys.path so we can import the cross-product helper
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'shared' / 'helpers' / 'canonical_join_keys.py').exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError('Run this notebook from the vital-statistics-harmonization repo root.')
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
from shared.helpers.canonical_join_keys import to_canonical_natality, NATALITY_TO_CANONICAL

print(f'Repo root: {REPO_ROOT}')
print(f'Cross-product rename map: {NATALITY_TO_CANONICAL}')

Repo root: /Users/yoelplutchok/Desktop/vital-statistics-harmonization
Cross-product rename map: {'year': 'data_year', 'restatus': 'residence_status', 'maternal_race_bridged4': 'maternal_race_bridged', 'maternal_hispanic_origin': 'hispanic_origin'}


## Section 0 — Load all three parquets, apply each canonical filter

Demonstrates the unified-resource claim: all three products load with consistent
demographic columns after the helper's read-time rename, and each product's
canonical filter is applied at load time.

In [2]:
# --- Natality (v2.7.0 harmonized + derived) ---
NAT_PARQUET = '/Users/yoelplutchok/Desktop/natality-harmonization/output/harmonized/natality_v2_harmonized_derived.parquet'
nat = pd.read_parquet(NAT_PARQUET, columns=['year', 'restatus', 'maternal_age', 'maternal_race_bridged4'])
nat = to_canonical_natality(nat)  # restatus -> residence_status, year -> data_year, etc.
nat_resident = nat[nat['residence_status'] != 4]  # int filter — natality side
print(f'Natality total: {len(nat):,}; resident: {len(nat_resident):,} (after restatus != 4)')
del nat  # release memory

Natality total: 138,819,655; resident: 138,582,904 (after restatus != 4)


In [3]:
# --- Linked birth–infant death (v3 derived) ---
LINKED_PARQUET = '/Users/yoelplutchok/Desktop/natality-harmonization/output/harmonized/natality_v3_linked_harmonized_derived.parquet'
linked = pd.read_parquet(LINKED_PARQUET, columns=['year', 'restatus'])
linked_resident = linked[linked['restatus'] != 4]
print(f'Linked total: {len(linked):,}; resident: {len(linked_resident):,} (after restatus != 4)')
del linked, linked_resident  # not used downstream in this notebook — just demonstrating load

Linked total: 74,943,824; resident: 74,785,708 (after restatus != 4)


In [4]:
# --- Fetal death (v2.0.0 derived) ---
FD_PARQUET = '/Users/yoelplutchok/Desktop/fetal-death-harmonization/fetal_death_derived.parquet'
fd = pd.read_parquet(
    FD_PARQUET,
    columns=['data_year', 'tabulation_flag', 'residence_status', 'maternal_age', 'maternal_race_bridged', 'hispanic_origin'],
)
# String filter on fetal-death side (parquet stores as object dtype, see notebook intro)
fd_nvsr = fd[(fd['tabulation_flag'] == '2') & (fd['residence_status'] != '4')]
print(f'Fetal-death total: {len(fd):,}; NVSR-pop (tab_flag==2 AND res!=4): {len(fd_nvsr):,}')

Fetal-death total: 1,634,195; NVSR-pop (tab_flag==2 AND res!=4): 727,155


## Section A — 2022 fetal mortality rate by maternal age band (NVSR 73-09 Table 4)

*NVSR 73-09* (Fetal Mortality: United States, 2022) Table 4 publishes 2022 fetal
deaths broken out by 8 maternal age bands: `<15`, `15-19`, `20-24`, `25-29`,
`30-34`, `35-39`, `40-44`, `45+`. The 8 cells are pre-encoded in
`fetal_death/external_validation_targets.csv` and reproduced here byte-exact from
the harmonized parquet; the joint-use denominator is recomputed from the natality
harmonized parquet under the same age binning and `restatus != 4` filter.

In [5]:
# --- Numerator: 2022 fetal deaths by NVSR age band ---
fd_2022 = fd_nvsr[fd_nvsr['data_year'] == 2022].copy()
fd_2022['maternal_age_int'] = pd.to_numeric(fd_2022['maternal_age'], errors='coerce')
assert len(fd_2022) == 20202, f'Unexpected 2022 NVSR-pop: {len(fd_2022)}'

NVSR_BANDS = [
    ('<15',   lambda a: a < 15),
    ('15-19', lambda a: (a >= 15) & (a <= 19)),
    ('20-24', lambda a: (a >= 20) & (a <= 24)),
    ('25-29', lambda a: (a >= 25) & (a <= 29)),
    ('30-34', lambda a: (a >= 30) & (a <= 34)),
    ('35-39', lambda a: (a >= 35) & (a <= 39)),
    ('40-44', lambda a: (a >= 40) & (a <= 44)),
    ('45+',   lambda a: a >= 45),
]
fd_by_band = {name: int(pred(fd_2022['maternal_age_int']).sum()) for name, pred in NVSR_BANDS}
pd.Series(fd_by_band, name='fetal_deaths_2022')

<15        16
15-19     991
20-24    3631
25-29    5071
30-34    5634
35-39    3613
40-44    1138
45+       108
Name: fetal_deaths_2022, dtype: int64

In [6]:
# --- Denominator: 2022 live births by NVSR age band (from natality) ---
nat_2022 = pd.read_parquet(NAT_PARQUET, columns=['year', 'restatus', 'maternal_age'])
nat_2022 = nat_2022[(nat_2022['year'] == 2022) & (nat_2022['restatus'] != 4)]
assert len(nat_2022) == 3667758, f'Unexpected 2022 resident-natality count: {len(nat_2022)}'
lb_by_band = {name: int(pred(nat_2022['maternal_age']).sum()) for name, pred in NVSR_BANDS}
pd.Series(lb_by_band, name='live_births_2022')

<15         1825
15-19     143789
20-24     638685
25-29    1013417
30-34    1118787
35-39     606598
40-44     134115
45+        10542
Name: live_births_2022, dtype: int64

In [7]:
# --- Validation table vs NVSR 73-09 Table 4 (pre-encoded targets) ---
NVSR_TARGETS = {
    '<15': 16, '15-19': 991, '20-24': 3631, '25-29': 5071,
    '30-34': 5634, '35-39': 3613, '40-44': 1138, '45+': 108,
}
rows = []
for band_name in [b[0] for b in NVSR_BANDS]:
    fd_n = fd_by_band[band_name]
    lb_n = lb_by_band[band_name]
    target = NVSR_TARGETS[band_name]
    diff = fd_n - target
    fmr = 1000 * fd_n / (lb_n + fd_n) if (lb_n + fd_n) > 0 else float('nan')
    rows.append({
        'age_band': band_name, 'fetal_deaths': fd_n, 'NVSR_73-09_T4': target,
        'diff': diff, 'status': 'PASS' if diff == 0 else f'DIFF{diff:+}',
        'live_births': lb_n, 'FMR_per_1000': round(fmr, 2),
    })
section_a = pd.DataFrame(rows)
section_a

,age_band,fetal_deaths,NVSR_73-09_T4,diff,status,live_births,FMR_per_1000
0,<15,16,16,0,PASS,1825,8.69
1,15-19,991,991,0,PASS,143789,6.84
2,20-24,3631,3631,0,PASS,638685,5.65
3,25-29,5071,5071,0,PASS,1013417,4.98
4,30-34,5634,5634,0,PASS,1118787,5.01
5,35-39,3613,3613,0,PASS,606598,5.92
6,40-44,1138,1138,0,PASS,134115,8.41
7,45+,108,108,0,PASS,10542,10.14


In [8]:
# --- Aggregate FMR (sanity: should round to NVSR-published 5.48) ---
num = sum(fd_by_band.values())
den = sum(lb_by_band.values()) + num
agg_fmr = 1000 * num / den
print(f'Aggregate 2022 FMR (sum-of-bands): {agg_fmr:.4f} per 1,000 (LB+FD)')
print(f'NVSR 73-09 Table 1 published rate: 5.48 per 1,000')
print(f'|diff| = {abs(agg_fmr - 5.48):.4f}; tolerance = 0.01 (rounding)')
assert abs(agg_fmr - 5.48) < 0.01, 'Aggregate FMR drift exceeds rounding tolerance'
print('PASS')

Aggregate 2022 FMR (sum-of-bands): 5.4778 per 1,000 (LB+FD)
NVSR 73-09 Table 1 published rate: 5.48 per 1,000
|diff| = 0.0022; tolerance = 0.01 (rounding)
PASS


**Section A result.** All 8 NVSR 73-09 Table 4 age cells reproduce byte-exact from
the harmonized parquet (Diff=0 across the board), aggregate FMR matches the
published per-1,000 rate within rounding noise (5.4778 vs 5.48), and the per-band
rates show the expected U-shaped age–FMR relationship (highest at the under-15
and 45+ tails, lowest at 25–29).

This is the manuscript's strongest reproducibility claim for the joint-use layer:
the cross-product machinery reproduces a published NVSR table at the cell level
with zero record drift.

## Section B — 2017 fetal mortality rate by maternal race (machinery demo)

2017 is the last year `maternal_race_bridged` is non-null in both products. NCHS
dropped MBRACE from the natality public-use file starting 2020 and from the fetal-
death public-use file starting 2018; bridged-race-stratified joint-use is
therefore limited to 1992–2002 + 2005–2017 (24 years) with current data.
Reconciling `maternal_race_ethnicity_5` (natality 2020+) and `race_hispanic_revised`
(fetal-death 2014+) to extend race-stratified joint-use to 2018–2022 is future
work.

This section demonstrates the joint-use machinery on 2017 race data using both
denominator paths: (a) the pre-built `stratified_denominators.csv`, and (b)
direct recompute from the natality parquet. Both paths produce identical counts
(Task 1 verified). *NVSR* cell-level validation of these race-stratified rates is
deferred to the paper companion notebook (Task 4) to avoid PDF-transcription risk
in this notebook's scope.

In [9]:
# --- Numerator: 2017 fetal deaths by maternal_race_bridged ---
fd_2017 = fd_nvsr[fd_nvsr['data_year'] == 2017]
assert len(fd_2017) == 22827, f'Unexpected 2017 NVSR-pop: {len(fd_2017)}'
fd_by_race = fd_2017.groupby('maternal_race_bridged', dropna=False).size().sort_index()
fd_by_race.name = 'fetal_deaths_2017'
fd_by_race

maternal_race_bridged
1    14603
2     6636
3      305
4     1283
Name: fetal_deaths_2017, dtype: int64

In [10]:
# --- Denominator path (a): from the pre-built stratified denominators CSV ---
STRAT_CSV = '/Users/yoelplutchok/Desktop/vital-statistics-harmonization/fetal_death/stratified_denominators.csv'
denom = pd.read_csv(STRAT_CSV)
lb_by_race_csv = (
    denom[denom['data_year'] == 2017]
    .groupby('maternal_race_bridged', dropna=False)['live_births']
    .sum()
    .sort_index()
)
lb_by_race_csv.name = 'live_births_2017_via_csv'
lb_by_race_csv

maternal_race_bridged
1.0    2857845
2.0     658115
3.0      41916
4.0     297624
Name: live_births_2017_via_csv, dtype: int64

In [11]:
# --- Denominator path (b): direct natality recompute (cross-check Task 1's CSV) ---
nat_2017 = pd.read_parquet(
    NAT_PARQUET, columns=['year', 'restatus', 'maternal_race_bridged4'],
)
nat_2017 = to_canonical_natality(nat_2017)
nat_2017 = nat_2017[(nat_2017['data_year'] == 2017) & (nat_2017['residence_status'] != 4)]
lb_by_race_direct = (
    nat_2017.groupby('maternal_race_bridged', dropna=False).size().sort_index()
)
lb_by_race_direct.name = 'live_births_2017_via_parquet'
# Cross-check (Task 1 receipt criterion C: race × year independent path):
consistent = (lb_by_race_csv == lb_by_race_direct).all()
print(f'CSV and direct paths agree on 2017 race-stratified live-birth counts: {consistent}')
assert consistent, 'Cross-check FAIL — stratified_denominators.csv and direct parquet recompute diverge'
pd.DataFrame({'csv_path': lb_by_race_csv, 'direct_path': lb_by_race_direct})

CSV and direct paths agree on 2017 race-stratified live-birth counts: True


,csv_path,direct_path
maternal_race_bridged,,
1.0,2857845,2857845
2.0,658115,658115
3.0,41916,41916
4.0,297624,297624


In [12]:
# --- Fetal mortality rate by race, 2017 ---
RACE_LABELS = {1: 'White', 2: 'Black', 3: 'AIAN', 4: 'Asian/PI'}
rows = []
for race_code in [1, 2, 3, 4]:
    fd_n = int(fd_by_race.get(str(race_code), 0))
    lb_n = int(lb_by_race_csv.get(float(race_code), 0))
    fmr = 1000 * fd_n / (lb_n + fd_n) if (lb_n + fd_n) > 0 else float('nan')
    rows.append({
        'race': RACE_LABELS[race_code], 'code': race_code,
        'fetal_deaths': fd_n, 'live_births': lb_n,
        'FMR_per_1000': round(fmr, 2),
    })
section_b = pd.DataFrame(rows)
section_b

,race,code,fetal_deaths,live_births,FMR_per_1000
0,White,1,14603,2857845,5.08
1,Black,2,6636,658115,9.98
2,AIAN,3,305,41916,7.22
3,Asian/PI,4,1283,297624,4.29


**Section B result.** The race-stratified joint-use machinery reproduces the
expected demographic pattern: the Black maternal-race stratum carries roughly
double the per-1,000 FMR of the White stratum, a long-documented U.S. perinatal-
epidemiology pattern. The two denominator paths (pre-built CSV vs direct natality
recompute) agree cell-by-cell (Task 1 verified this; the assertion in the cell
above is the cross-check).

*NVSR* cell-level validation of these race-stratified rates is deferred to the
paper companion notebook (Task 4 in `NEXT_STEPS.md` §15).

## Pass / fail summary

| Check | Outcome |
|---|---|
| Natality + linked + fetal-death parquets all load with canonical filters | PASS |
| Section A: 8/8 NVSR 73-09 Table 4 age cells byte-exact | PASS |
| Section A aggregate FMR 5.4778 within rounding tolerance of NVSR-published 5.48 | PASS |
| Section A row-count conservation (numerator + denominator both sides) | PASS |
| Section B: CSV denominator path agrees with direct natality recompute, cell-by-cell | PASS |
| Section B: race-stratified FMR pattern (Black vs White ~2×) reproduces published epidemiology | PASS |
| Section B *NVSR* cell-level validation | DEFERRED to Task 4 (paper companion) |

**No FAILs.** The joint-use layer reproduces a published NVSR table at the cell
level and exposes the race-stratified machinery on the last year of bridged-race
availability.